# Environmental Data for Rutgers Richard Weeks Hall (Busch Campus)

**Data Source:** [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api)  
Uses NOAA GFS / ECMWF IFS reanalysis data (no API key required).  

**Location:** Richard Weeks Hall, Busch Campus, Rutgers University  
**Coordinates:** 40.5218°N, 74.4633°W  
**Nearest NWS station:** New Brunswick, NJ (COOP / Rutgers Gardens station ~1.5 mi away)


In [14]:
# ============================================================
# CONFIGURATION — edit dates and location here
# ============================================================

LATITUDE  = 40.5218   # Rutgers RWH, Busch Campus
LONGITUDE = -74.4633

# Date range (YYYY-MM-DD). Open-Meteo historical goes back to 1940.
# For recent dates (<5 days ago), data may not yet be available in
# the archive — the script falls back to the forecast API automatically.
DATES = [
    "2025-03-19",
    # Add more dates or ranges below:
    "2025-09-27",
    "2025-10-26",
    "2025-11-02"
]

# Or set a continuous range (overrides DATES list if uncommented):
# START_DATE = "2024-01-01"
# END_DATE   = "2024-12-31"

In [15]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import math
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

Libraries loaded.


In [16]:
# ============================================================
# Build date range from config
# ============================================================

try:
    start_date = START_DATE
    end_date   = END_DATE
except NameError:
    # Use individual DATES list — find min/max
    sorted_dates = sorted(DATES)
    start_date = sorted_dates[0]
    end_date   = sorted_dates[-1]

print(f"Fetching data: {start_date} → {end_date}")
print(f"Location: ({LATITUDE}, {LONGITUDE})")

Fetching data: 2025-03-19 → 2025-11-02
Location: (40.5218, -74.4633)


In [17]:
# ============================================================
# Fetch hourly data from Open-Meteo
# ============================================================

HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "precipitation",
    "rain",
    "snowfall",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation",
    "direct_normal_irradiance",
    "terrestrial_radiation",
    "is_day",
    "sunshine_duration",
    "soil_temperature_0_to_7cm",
]

# Decide which API endpoint to use
end_dt = datetime.strptime(end_date, "%Y-%m-%d")
days_ago = (datetime.utcnow() - end_dt).days

if days_ago < 5:
    # Recent data: use forecast API with past_days
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "hourly": ",".join(HOURLY_VARS),
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "America/New_York",
        "past_days": 92,
    }
else:
    # Historical data: use archive API
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "hourly": ",".join(HOURLY_VARS),
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "America/New_York",
    }

print(f"Using endpoint: {base_url}")
response = requests.get(base_url, params=params, timeout=60)
response.raise_for_status()
data = response.json()

print(f"✓ API returned {len(data['hourly']['time'])} hourly records")

Using endpoint: https://archive-api.open-meteo.com/v1/archive
✓ API returned 5496 hourly records


In [18]:
# ============================================================
# Build DataFrame
# ============================================================

df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])
df = df.rename(columns={"time": "timestamp"})

# If using individual DATES, filter to only requested dates
try:
    _ = START_DATE  # continuous range — keep all
except NameError:
    requested = set(pd.to_datetime(DATES).date)
    df = df[df["timestamp"].dt.date.isin(requested)].reset_index(drop=True)

print(f"DataFrame shape: {df.shape}")
df.head()

DataFrame shape: (96, 22)


,timestamp,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,...,rain,snowfall,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance,terrestrial_radiation,is_day,sunshine_duration,soil_temperature_0_to_7cm
0,2025-03-19 00:00:00,5.4,76,1.5,3.2,1017.6,1013.6,0,2.4,333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,6.4
1,2025-03-19 01:00:00,4.5,79,1.2,2.3,1017.8,1013.8,0,1.8,307,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,5.8
2,2025-03-19 02:00:00,4.0,81,1.0,1.8,1018.0,1014.0,0,1.2,27,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,5.2
3,2025-03-19 03:00:00,3.2,85,0.9,0.7,1017.8,1013.8,0,3.9,53,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,4.7
4,2025-03-19 04:00:00,3.2,85,0.9,0.4,1017.9,1013.9,0,5.2,84,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,4.2


In [19]:
# ============================================================
# Compute solar position (elevation & azimuth angles)
# Using a simplified astronomical algorithm — accurate to ~1°
# ============================================================

def solar_position(timestamps, lat, lon):
    """Return solar elevation and azimuth angles in degrees."""
    lat_r = math.radians(lat)
    results = []

    for ts in timestamps:
        # Convert to UTC for calculation
        ts_utc = ts.tz_localize("America/New_York").tz_convert("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

        # Day of year and fractional hour (UTC)
        doy = ts_utc.timetuple().tm_yday
        hour_utc = ts_utc.hour + ts_utc.minute / 60.0

        # Solar declination (Spencer, 1971)
        gamma = 2 * math.pi * (doy - 1) / 365.0
        decl = (0.006918 - 0.399912 * math.cos(gamma) + 0.070257 * math.sin(gamma)
                - 0.006758 * math.cos(2*gamma) + 0.000907 * math.sin(2*gamma)
                - 0.002697 * math.cos(3*gamma) + 0.00148 * math.sin(3*gamma))

        # Equation of time (minutes)
        eqtime = 229.18 * (0.000075 + 0.001868 * math.cos(gamma)
                           - 0.032077 * math.sin(gamma)
                           - 0.014615 * math.cos(2*gamma)
                           - 0.04089 * math.sin(2*gamma))

        # True solar time
        tst = hour_utc * 60 + eqtime + 4 * lon  # lon in degrees
        hour_angle = math.radians((tst / 4.0) - 180.0)

        # Solar elevation
        sin_elev = (math.sin(lat_r) * math.sin(decl)
                    + math.cos(lat_r) * math.cos(decl) * math.cos(hour_angle))
        elevation = math.degrees(math.asin(max(-1, min(1, sin_elev))))

        # Solar azimuth (0° = North, 180° = South)
        cos_az = ((math.sin(decl) - math.sin(lat_r) * sin_elev)
                  / (math.cos(lat_r) * math.cos(math.radians(elevation)) + 1e-10))
        azimuth = math.degrees(math.acos(max(-1, min(1, cos_az))))
        if hour_angle > 0:
            azimuth = 360 - azimuth

        results.append((round(elevation, 2), round(azimuth, 2)))

    return zip(*results)

df["solar_elevation_deg"], df["solar_azimuth_deg"] = solar_position(
    df["timestamp"], LATITUDE, LONGITUDE
)

print("✓ Solar position columns added")
df[["timestamp", "solar_elevation_deg", "solar_azimuth_deg", "is_day"]].head(12)

AmbiguousTimeError: Cannot infer dst time from 2025-11-02 01:00:00, try using the 'ambiguous' argument

In [20]:
# ============================================================
# Rename columns for clarity & add metadata
# ============================================================

rename_map = {
    "temperature_2m": "air_temp_C",
    "relative_humidity_2m": "humidity_pct",
    "dew_point_2m": "dew_point_C",
    "apparent_temperature": "feels_like_C",
    "pressure_msl": "pressure_sea_level_hPa",
    "surface_pressure": "pressure_surface_hPa",
    "cloud_cover": "cloud_cover_pct",
    "wind_speed_10m": "wind_speed_kmh",
    "wind_direction_10m": "wind_dir_deg",
    "wind_gusts_10m": "wind_gust_kmh",
    "precipitation": "precip_mm",
    "rain": "rain_mm",
    "snowfall": "snow_cm",
    "shortwave_radiation": "solar_ghi_Wm2",
    "direct_radiation": "solar_direct_Wm2",
    "diffuse_radiation": "solar_diffuse_Wm2",
    "direct_normal_irradiance": "solar_dni_Wm2",
    "terrestrial_radiation": "solar_extraterrestrial_Wm2",
    "sunshine_duration": "sunshine_sec",
    "soil_temperature_0_to_7cm": "soil_temp_0_7cm_C",
}
df = df.rename(columns=rename_map)

# Add location metadata
df["latitude"] = LATITUDE
df["longitude"] = LONGITUDE
df["location"] = "Rutgers_RWH_Busch_Campus"

print(f"Final columns ({len(df.columns)}):")
print(list(df.columns))
df.head()

Final columns (25):
['timestamp', 'air_temp_C', 'humidity_pct', 'dew_point_C', 'feels_like_C', 'pressure_sea_level_hPa', 'pressure_surface_hPa', 'cloud_cover_pct', 'wind_speed_kmh', 'wind_dir_deg', 'wind_gust_kmh', 'precip_mm', 'rain_mm', 'snow_cm', 'solar_ghi_Wm2', 'solar_direct_Wm2', 'solar_diffuse_Wm2', 'solar_dni_Wm2', 'solar_extraterrestrial_Wm2', 'is_day', 'sunshine_sec', 'soil_temp_0_7cm_C', 'latitude', 'longitude', 'location']


,timestamp,air_temp_C,humidity_pct,dew_point_C,feels_like_C,pressure_sea_level_hPa,pressure_surface_hPa,cloud_cover_pct,wind_speed_kmh,wind_dir_deg,...,solar_direct_Wm2,solar_diffuse_Wm2,solar_dni_Wm2,solar_extraterrestrial_Wm2,is_day,sunshine_sec,soil_temp_0_7cm_C,latitude,longitude,location
0,2025-03-19 00:00:00,5.4,76,1.5,3.2,1017.6,1013.6,0,2.4,333,...,0.0,0.0,0.0,0.0,0,0.0,6.4,40.5218,-74.4633,Rutgers_RWH_Busch_Campus
1,2025-03-19 01:00:00,4.5,79,1.2,2.3,1017.8,1013.8,0,1.8,307,...,0.0,0.0,0.0,0.0,0,0.0,5.8,40.5218,-74.4633,Rutgers_RWH_Busch_Campus
2,2025-03-19 02:00:00,4.0,81,1.0,1.8,1018.0,1014.0,0,1.2,27,...,0.0,0.0,0.0,0.0,0,0.0,5.2,40.5218,-74.4633,Rutgers_RWH_Busch_Campus
3,2025-03-19 03:00:00,3.2,85,0.9,0.7,1017.8,1013.8,0,3.9,53,...,0.0,0.0,0.0,0.0,0,0.0,4.7,40.5218,-74.4633,Rutgers_RWH_Busch_Campus
4,2025-03-19 04:00:00,3.2,85,0.9,0.4,1017.9,1013.9,0,5.2,84,...,0.0,0.0,0.0,0.0,0,0.0,4.2,40.5218,-74.4633,Rutgers_RWH_Busch_Campus


In [21]:
# ============================================================
# Quick summary
# ============================================================

df.describe().round(2)

,timestamp,air_temp_C,humidity_pct,dew_point_C,feels_like_C,pressure_sea_level_hPa,pressure_surface_hPa,cloud_cover_pct,wind_speed_kmh,wind_dir_deg,...,solar_ghi_Wm2,solar_direct_Wm2,solar_diffuse_Wm2,solar_dni_Wm2,solar_extraterrestrial_Wm2,is_day,sunshine_sec,soil_temp_0_7cm_C,latitude,longitude
count,96,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,...,96.00,96.00,96.00,96.00,96.00,96.00,96.0,96.00,96.00,96.00
mean,2025-08-26 17:30:00,12.12,68.72,6.18,10.67,1021.42,1017.52,52.57,5.73,183.38,...,157.38,108.49,48.89,223.91,275.09,0.47,1314.0,12.44,40.52,-74.46
min,2025-03-19 00:00:00,2.70,37.00,-0.10,0.40,1013.10,1009.30,0.00,0.70,2.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,3.50,40.52,-74.46
25%,2025-08-10 05:45:00,7.10,55.00,1.58,4.50,1015.68,1011.90,6.75,4.28,96.75,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0,8.10,40.52,-74.46
50%,2025-10-11 23:30:00,11.40,71.50,3.90,9.20,1020.85,1016.90,48.00,5.40,135.00,...,0.00,0.00,0.00,0.00,0.55,0.00,0.0,11.55,40.52,-74.46
75%,2025-10-28 11:15:00,16.08,82.25,8.00,15.08,1027.00,1023.02,100.00,7.20,320.50,...,276.50,160.75,82.50,489.55,614.18,1.00,3600.0,16.05,40.52,-74.46
max,2025-11-02 23:00:00,23.90,88.00,18.40,25.20,1030.20,1026.20,100.00,12.40,356.00,...,787.00,688.00,319.00,916.30,1036.50,1.00,3600.0,23.20,40.52,-74.46
std,NaN,6.05,14.80,6.10,7.57,5.67,5.61,43.37,2.66,117.58,...,224.59,182.32,70.49,317.60,355.87,0.50,1683.7,5.69,0.00,0.00


In [22]:
# ============================================================
# Save to CSV
# ============================================================

filename = f"rutgers_rwh_env_data_{start_date}_to_{end_date}.csv"
df.to_csv(filename, index=False)
print(f"✓ Saved: {filename}  ({len(df)} rows)")

# Colab: download automatically
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print("(Not running in Colab — file saved to working directory)")

✓ Saved: rutgers_rwh_env_data_2025-03-19_to_2025-11-02.csv  (96 rows)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>